# Group 1 — Basic Pitch Transcription

Run basic-pitch (ONNX backend) on a monophonic recording from a continuous
instrument (flute, sax, violin, voice), inspect the resulting MIDI, and
evaluate transcription quality.

**Prerequisites:** Install transcription dependencies (see cell below).

**Outputs:** A `.mid` file with note events (pitch, onset, offset, velocity).

In [ ]:
# Install dependencies (run once)
!pip install "basic-pitch[onnx]" onnxruntime librosa pretty_midi matplotlib

In [ ]:
import sys
from pathlib import Path

# Add notebooks/ to path so we can import from shared/
sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import pretty_midi
import IPython.display as ipd

from shared.transcription import transcribe_wav

# --------------- Configuration (edit these) ---------------
INPUT_WAV = "../../recordings/your_recording.wav"  # <-- edit this
OUTPUT_MID = "../../output/transcription_group1.mid"

In [ ]:
# ---- Run transcription ----

model_output, midi_data, note_events = transcribe_wav(INPUT_WAV, OUTPUT_MID)

In [ ]:
# ---- Inspect MIDI ----

pm = pretty_midi.PrettyMIDI(OUTPUT_MID)

print(f"Instruments: {len(pm.instruments)}")
for i, inst in enumerate(pm.instruments):
    print(f"  [{i}] {inst.name or 'unnamed'}: {len(inst.notes)} notes")

# Print first 20 notes
if pm.instruments:
    notes = pm.instruments[0].notes
    print(f"\nFirst {min(20, len(notes))} notes:")
    print(f"  {'Pitch':>5}  {'Start':>7}  {'End':>7}  {'Vel':>3}")
    for n in notes[:20]:
        name = pretty_midi.note_number_to_name(n.pitch)
        print(f"  {name:>5}  {n.start:7.3f}  {n.end:7.3f}  {n.velocity:3d}")

In [ ]:
# ---- Piano roll visualization ----

piano_roll = pm.get_piano_roll(fs=100)
plt.figure(figsize=(14, 4))
plt.imshow(
    piano_roll,
    aspect="auto",
    origin="lower",
    cmap="magma",
    extent=[0, piano_roll.shape[1] / 100, 0, 127],
)
plt.ylabel("MIDI Pitch")
plt.xlabel("Time (s)")
plt.title("Transcription Piano Roll")
plt.colorbar(label="Velocity")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Audio comparison ----

print("Original audio:")
ipd.display(ipd.Audio(INPUT_WAV))

# Synthesize MIDI back to audio for A/B comparison
midi_audio = pm.fluidsynth(fs=44100)
print("\nSynthesized from MIDI (FluidSynth):")
ipd.display(ipd.Audio(midi_audio, rate=44100))